# 10. Pull-Up Dedicated Pose Colab


## Reason, Approach, Result Interpretation

This stage starts the first dedicated pose follow-up beyond squat.

What this notebook is testing:
- whether `pull_up` improves when pose is tuned as a dedicated exercise branch instead of only using the shared `6B` setup
- whether a dedicated pose-tuned `pull_up` branch can beat the saved pose baseline and the saved RGB baselines on the same exercise

Important caution:
- this is **not** a squat-style custom feature-engineering branch
- this stage still uses the generic Stage 5 normalized pose sequences
- the specialization here is exercise-specific training and tuning, not new handcrafted pull-up features


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

TCN_TRAIN_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')
REGISTER_REL = Path('artifacts/3_Modeling/register_experiment.py')


def sync_drive_file(rel: Path) -> None:
    src = CODE_ROOT / rel
    dst = DRIVE_PROJECT_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not src.exists() and not dst.exists():
        print(f'[sync] missing both copies: {rel}')
        return
    if not src.exists():
        print(f'[sync] keeping Drive copy (no /content source): {rel}')
        return
    if not dst.exists():
        shutil.copy2(src, dst)
        print(f'[sync] copied /content -> Drive (Drive missing): {rel}')
        return
    if src.stat().st_mtime > dst.stat().st_mtime + 1.0:
        shutil.copy2(src, dst)
        print(f'[sync] copied newer /content -> Drive: {rel}')
    else:
        print(f'[sync] keeping Drive copy (newer or equal): {rel}')


for rel in [TCN_TRAIN_REL, COMPARE_REL, REGISTER_REL]:
    sync_drive_file(rel)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
POSE_INDEX = ANNOTATION_DIR / 'pose_sequence_index.csv'
TRAINING_OUTPUTS = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs'

print('POSE_INDEX =', POSE_INDEX)
print('Pose trainer exists =', (DRIVE_PROJECT_ROOT / TCN_TRAIN_REL).exists())
print('Compare script exists =', (DRIVE_PROJECT_ROOT / COMPARE_REL).exists())


In [ ]:
import pandas as pd

EXERCISE = 'pull_up'
SEQ_LEN = 192

BASELINE_RUNS = [
    {'source': 'pose_shared', 'run_name': 'pose_count_tcn_pull_up_seq192'},
    {'source': 'rgb_stage7', 'run_name': 'rgb_count_tcn_pull_up_seq192'},
    {'source': 'rgb_stronger', 'run_name': 'rgb_resnet50_count_tcn_pull_up_seq192'},
]

COMMON = {
    'epochs': '80',
    'batch_size': '16',
    'weight_decay': '0.0001',
    'kernel_size': '3',
    'num_blocks': '4',
    'loss': 'l1',
    'eval_transform': 'raw',
    'selection_metric': 'mae',
    'sampler': 'balanced_count',
    'time_warp_range': '0.12',
    'feature_noise_std': '0.02',
    'frame_dropout_prob': '0.03',
}

CANDIDATES = [
    {
        'run_name': 'pull_up_pose_tcn_l1_channels96',
        'lr': '0.001',
        'channels': '96',
        'dropout': '0.20',
        'patience': '15',
    },
    {
        'run_name': 'pull_up_pose_tcn_l1_channels96_dropout01',
        'lr': '0.001',
        'channels': '96',
        'dropout': '0.10',
        'patience': '15',
    },
    {
        'run_name': 'pull_up_pose_tcn_l1_channels96_lr5e4',
        'lr': '0.0005',
        'channels': '96',
        'dropout': '0.20',
        'patience': '15',
    },
    {
        'run_name': 'pull_up_pose_tcn_l1_channels128',
        'lr': '0.001',
        'channels': '128',
        'dropout': '0.20',
        'patience': '15',
    },
]

meta_df = pd.read_csv(POSE_INDEX)
exercise_df = meta_df[meta_df['type'] == EXERCISE].copy()
display(exercise_df.groupby('split').size().rename(EXERCISE).to_frame().T)
display(pd.DataFrame(CANDIDATES))


In [ ]:
import subprocess

training_failures = []
for cfg in CANDIDATES:
    cmd = [
        'python', '-u', str(DRIVE_PROJECT_ROOT / TCN_TRAIN_REL),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--index-csv', str(POSE_INDEX),
        '--run-name', cfg['run_name'],
        '--exercise', EXERCISE,
        '--seq-len', str(SEQ_LEN),
        '--epochs', COMMON['epochs'],
        '--batch-size', COMMON['batch_size'],
        '--lr', cfg['lr'],
        '--weight-decay', COMMON['weight_decay'],
        '--channels', cfg['channels'],
        '--kernel-size', COMMON['kernel_size'],
        '--num-blocks', COMMON['num_blocks'],
        '--dropout', cfg['dropout'],
        '--patience', cfg['patience'],
        '--loss', COMMON['loss'],
        '--eval-transform', COMMON['eval_transform'],
        '--selection-metric', COMMON['selection_metric'],
        '--sampler', COMMON['sampler'],
        '--time-warp-range', COMMON['time_warp_range'],
        '--feature-noise-std', COMMON['feature_noise_std'],
        '--frame-dropout-prob', COMMON['frame_dropout_prob'],
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        training_failures.append({
            'exercise': EXERCISE,
            'run_name': cfg['run_name'],
            'returncode': exc.returncode,
        })
        print(f"FAILED: {cfg['run_name']} (returncode={exc.returncode})")

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All dedicated pull_up pose runs completed.')


In [ ]:
import json

rows = []
for spec in BASELINE_RUNS + [{'source': 'candidate', 'run_name': cfg['run_name']} for cfg in CANDIDATES]:
    metrics_path = TRAINING_OUTPUTS / spec['run_name'] / 'metrics_summary.json'
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text())
    rows.append({
        'source': spec['source'],
        'run_name': spec['run_name'],
        'best_epoch': metrics['best_epoch'],
        'train_mae': metrics['train_metrics']['mae'],
        'train_within_1': metrics['train_metrics']['within_1'],
        'valid_mae': metrics['valid_metrics']['mae'],
        'valid_rmse': metrics['valid_metrics']['rmse'],
        'valid_within_1': metrics['valid_metrics']['within_1'],
    })

metrics_df = pd.DataFrame(rows)
if metrics_df.empty:
    print('No metrics found yet. Run the training cell first.')
else:
    pose_baseline_df = metrics_df.loc[metrics_df['run_name'] == 'pose_count_tcn_pull_up_seq192']
    rgb_stronger_df = metrics_df.loc[metrics_df['run_name'] == 'rgb_resnet50_count_tcn_pull_up_seq192']
    if pose_baseline_df.empty or rgb_stronger_df.empty:
        print('Missing one or both saved pull_up baselines; showing raw metrics only.')
        display(
            metrics_df.sort_values(['valid_mae', 'valid_within_1'], ascending=[True, False])[
                ['source', 'run_name', 'best_epoch', 'valid_mae', 'valid_rmse', 'valid_within_1']
            ]
        )
    else:
        pose_baseline = pose_baseline_df.iloc[0]
        rgb_stronger = rgb_stronger_df.iloc[0]
        metrics_df['delta_valid_mae_vs_pose'] = metrics_df['valid_mae'] - float(pose_baseline['valid_mae'])
        metrics_df['delta_valid_within_1_vs_pose'] = metrics_df['valid_within_1'] - float(pose_baseline['valid_within_1'])
        metrics_df['delta_valid_mae_vs_rgb_stronger'] = metrics_df['valid_mae'] - float(rgb_stronger['valid_mae'])
        metrics_df['delta_valid_within_1_vs_rgb_stronger'] = metrics_df['valid_within_1'] - float(rgb_stronger['valid_within_1'])
        display(
            metrics_df.sort_values(['valid_mae', 'valid_within_1'], ascending=[True, False])[
                [
                    'source',
                    'run_name',
                    'best_epoch',
                    'valid_mae',
                    'valid_rmse',
                    'valid_within_1',
                    'delta_valid_mae_vs_pose',
                    'delta_valid_within_1_vs_pose',
                    'delta_valid_mae_vs_rgb_stronger',
                    'delta_valid_within_1_vs_rgb_stronger',
                ]
            ]
        )


In [ ]:
comparison_failures = []
for cfg in CANDIDATES:
    run_dir = TRAINING_OUTPUTS / cfg['run_name']
    pred_csv = run_dir / 'predictions.csv'
    if not pred_csv.exists():
        continue
    output_json = run_dir / 'baseline_comparison_pull_up.json'
    output_csv = run_dir / 'baseline_comparison_pull_up.csv'
    cmd = [
        'python', '-u', str(DRIVE_PROJECT_ROOT / COMPARE_REL),
        '--index-csv', str(POSE_INDEX),
        '--predictions-csv', str(pred_csv),
        '--exercise', EXERCISE,
        '--output-json', str(output_json),
        '--output-csv', str(output_csv),
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        comparison_failures.append({
            'run_name': cfg['run_name'],
            'returncode': exc.returncode,
        })
        print(f"FAILED baseline comparison: {cfg['run_name']} (returncode={exc.returncode})")

comparison_rows = []
for cfg in CANDIDATES:
    summary_path = TRAINING_OUTPUTS / cfg['run_name'] / 'baseline_comparison_pull_up.json'
    if not summary_path.exists():
        continue
    summary = json.loads(summary_path.read_text())
    delta = summary.get('delta_vs_baseline') or summary.get('delta_metrics') or {}
    row_level = summary.get('row_level', {})
    comparison_rows.append({
        'run_name': cfg['run_name'],
        'model_mae': summary['model_metrics']['mae'],
        'baseline_mae': summary['baseline_metrics']['mae'],
        'delta_mae_vs_trivial': delta.get('mae'),
        'model_within_1': summary['model_metrics']['within_1'],
        'baseline_within_1': summary['baseline_metrics']['within_1'],
        'delta_within_1_vs_trivial': delta.get('within_1'),
        'model_beats_baseline_rows': row_level.get('model_beats_baseline', summary.get('model_beats_baseline_rows')),
        'valid_rows': row_level.get('valid_rows', summary.get('valid_rows')),
    })

if comparison_failures:
    display(pd.DataFrame(comparison_failures))
if comparison_rows:
    display(pd.DataFrame(comparison_rows).sort_values('model_mae'))
else:
    print('No baseline comparison summaries found yet.')


In [ ]:
print('Register the selected best run after review, for example:')
print(
    'python',
    DRIVE_PROJECT_ROOT / REGISTER_REL,
    '--stage', '10_pull_up_dedicated_pose',
    '--scope', 'pull_up',
    '--question', 'Does dedicated pose tuning on shared sequences improve pull_up counting?',
    '--decision', 'pending_review',
    '--artifact-reference', 'artifacts/3_Modeling/10_PullUp_Dedicated_Pose_Colab.ipynb',
)
